# Notebook 01: Feature Engineering

**Persona:** Data Engineer

**Purpose:** Build a comprehensive Feature Store for two ML use cases:
- **Session Conversion Prediction** (will a session result in a purchase?)
- **User Churn Prediction** (will a user become inactive?)

This notebook demonstrates:
1. Entity registration (simple, compound, hierarchical)
2. View-based Feature Views (static/slow-changing data)
3. Dynamic Table Feature Views (incremental refresh)
4. Tiled/Aggregation API Feature Views (time-windowed aggregations)
5. Chained Feature Views (DT reading from upstream DTs)
6. External pipeline integration (dbt-managed tables)
7. Wide/OBJECT Feature Views (packed semi-structured features)
8. VECTOR embeddings via Cortex
9. Feature View versioning

**Prerequisites:** Run Notebook 00 first

## 1. Setup & Connect

In [ ]:
import os, sys

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    sys.path.insert(0, ".")
    from feature_definitions.config import get_session, ROLES
    session = get_session(role=ROLES["dev"])

sys.path.insert(0, ".") if "." not in sys.path else None

from feature_definitions.config import get_config, ROLES
from snowflake.ml.feature_store import FeatureStore, CreationMode

cfg = get_config("DEV")
refresh_wh = cfg.get("refresh_warehouse", "FS_REFRESH_WH")
session.sql(f"USE ROLE {ROLES['dev']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

fs = FeatureStore(
    session=session,
    database=cfg["database"],
    name=cfg["fs_schema"],
    default_warehouse=refresh_wh,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print(f"Feature Store: {cfg['database']}.{cfg['fs_schema']} (DT warehouse={refresh_wh})")
print(f"Role: {session.get_current_role()}")

## 2. Register Entities

Entities define the join keys that connect Feature Views to training spines. Our clickstream model has a rich entity hierarchy:

| Entity | Join Key(s) | Type |
|---|---|---|
| VISITOR | VISITOR_ID | Simple |
| USER | USER_ID | Simple |
| HOUSEHOLD | HOUSEHOLD_ID | Simple |
| SESSION | SESSION_ID | Simple |
| PRODUCT | PRODUCT_ID | Simple |
| CATEGORY | CATEGORY_ID | Simple |
| SUPPLIER | SUPPLIER_ID | Simple |
| PRODUCT_SUPPLIER | PRODUCT_ID + SUPPLIER_ID | **Compound** |

In [ ]:
from feature_definitions.entities import register_all

entities = register_all(fs)
for name, entity in entities.items():
    keys = ", ".join(entity.join_keys)
    print(f"  {name:25s}  join_keys=[{keys}]")

print(f"\nTotal entities: {fs.list_entities().count()}")

## 3. Feature Views: View-Based (Static / Slow-Changing)

View-based Feature Views have `refresh_freq=None`. They execute the SQL at query time -- ideal for dimension tables and user profiles that change infrequently.

In [ ]:
from feature_definitions.shared_features import user_profile_features
from feature_definitions.conversion_features import product_catalog_features
from feature_definitions.compound_features import product_supplier_metrics

# User profile – slow-changing attributes
fv = user_profile_features(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"USER_PROFILE_FEATURES V01: {fv_reg.status}")
print(f"  Features: {[c for c in fv_reg.feature_df.columns if c not in ('USER_ID','UPDATED_TS')]}")

# Product catalog – joined with categories
fv = product_catalog_features(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"\nPRODUCT_CATALOG_FEATURES V01: {fv_reg.status}")

# Compound entity – product-supplier relationship
fv = product_supplier_metrics(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"\nPRODUCT_SUPPLIER_METRICS V01: {fv_reg.status}")
print(f"  Compound join keys: {[e.join_keys for e in fv_reg.entities]}")

In [ ]:
# Preview the user profile features
fv = fs.get_feature_view("USER_PROFILE_FEATURES", "V01")
fv.feature_df.limit(5).show()

## 4. Feature Views: Dynamic Table (Incremental Refresh)

DT-based Feature Views materialise results and refresh automatically at the configured `refresh_freq`. The Feature Store creates the underlying Dynamic Table and manages its lifecycle.

In [ ]:
from feature_definitions.conversion_features import session_behavior_features
from feature_definitions.churn_features import user_recency_raw, user_recency_features

# Session-level behavioural signals (conversion model)
fv = session_behavior_features(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"SESSION_BEHAVIOR_FEATURES V01: {fv_reg.status}")
print(f"  Refresh: {fv_reg.refresh_freq}")

# Recency features – days-since metrics (churn model)
# Raw recency timestamps (DT base, PIT-correct)
fv = user_recency_raw(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"\nUSER_RECENCY_RAW V01: {fv_reg.status}")

# View-based derivation: DAYS_SINCE via CURRENT_TIMESTAMP (serving only)
fv = user_recency_features(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"USER_RECENCY_FEATURES V01 (View): {fv_reg.status}")

## 5. Feature Views: Aggregation API (Tiled)

The Aggregation API creates *tiled* Feature Views that pre-compute partial aggregates in time buckets. At query time, tiles are assembled into the requested time windows (7d, 30d, 90d, etc.).

This is the most powerful pattern for time-windowed features -- efficient, PIT-correct, and incrementally maintained.

In [ ]:
from feature_definitions.shared_features import user_purchase_aggregates, user_session_engagement
from feature_definitions.conversion_features import user_engagement_realtime
from feature_definitions.churn_features import user_trend_features

# Purchase aggregations (shared by both models)
fv = user_purchase_aggregates(session, "DEV", version="V03")
fv_reg = fs.register_feature_view(feature_view=fv, version="V03", block=True)
print(f"USER_PURCHASE_AGGREGATES V03: {fv_reg.status}")

# Session engagement (shared)
fv = user_session_engagement(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"USER_SESSION_ENGAGEMENT V01: {fv_reg.status}")

# Hourly engagement – near-real-time (conversion)
fv = user_engagement_realtime(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"USER_ENGAGEMENT_REALTIME V01: {fv_reg.status}")

# Long-window trends (churn)
fv = user_trend_features(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"USER_TREND_FEATURES V01: {fv_reg.status}")

## 6. Two-Tier Recency Architecture (PIT-Correct Pattern)

The recency pipeline uses a two-tier architecture for Point-in-Time correctness:

- **`USER_RECENCY_RAW`** (DT) stores raw timestamps (`LAST_ORDER_TS`, `LAST_SESSION_TS`, etc.) which are PIT-correct when retrieved via ASOF.
- **`USER_RECENCY_FEATURES`** (View) derives `DAYS_SINCE_*` at query time via `CURRENT_TIMESTAMP()` for real-time serving.
- For **training data**, use `enrichment.derive_temporal_features()` post-retrieval with the spine timestamp (see NB 02).

This replaces the previous approach where `CURRENT_TIMESTAMP()` was baked into the DT, making DAYS_SINCE values silently incorrect for historical training data.

In [ ]:
# Verify the two-tier architecture
fv_raw  = fs.get_feature_view("USER_RECENCY_RAW", "V01")
fv_view = fs.get_feature_view("USER_RECENCY_FEATURES", "V01")

print("USER_RECENCY_RAW (DT base):")
print(f"  Status: {fv_raw.status}")
print(f"  Columns: {fv_raw.feature_df.columns}")

print("\nUSER_RECENCY_FEATURES (View derivation):")
print(f"  Status: {fv_view.status}")
print(f"  Columns: {fv_view.feature_df.columns}")

# RFM is no longer a separate FV. The churn model consumes raw components:
#   Recency  -> LAST_ORDER_TS from USER_RECENCY_RAW
#   Frequency -> ORDER_ID_CNT_30D from USER_PURCHASE_AGGREGATES
#   Monetary  -> TOTAL_AMT_SUM_30D from USER_PURCHASE_AGGREGATES
print("\nRFM components are served by existing FVs (no separate RFM FV needed)")

## 7. External Pipeline: dbt Integration

This pattern places a **View-based** Feature View on top of a table managed by an external tool (dbt, Airflow, etc.). The Feature Store owns the *feature definition*; the external tool owns the *transformation pipeline*.

Here we simulate the dbt output table and register a Feature View on it.

In [ ]:
from feature_definitions.dbt_features import create_simulated_dbt_table, user_order_summary_dbt

# Simulate the table a dbt model would produce
create_simulated_dbt_table(session, "DEV")
print("Created simulated dbt output: USER_ORDER_SUMMARY")

# Register View-based FV on the dbt-managed table
fv = user_order_summary_dbt(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"\nUSER_ORDER_SUMMARY_DBT V01: {fv_reg.status}")
fv_reg.feature_df.limit(3).show()

## 8. Wide Feature View: OBJECT Encapsulation

When a Feature View produces many sparse columns (e.g. interaction permutations), packing them into a single `OBJECT` column reduces schema complexity. The features are unpacked on the client side during preprocessing.

This FV computes per-user counts of (category × device) and (utm_source × device) interactions.

In [ ]:
from feature_definitions.wide_features import user_permutation_features_wide

fv = user_permutation_features_wide(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"USER_PERMUTATION_FEATURES_WIDE V01: {fv_reg.status}")

# Preview – note the FEATURES_OBJ column
import json as _json
sample = fv_reg.feature_df.limit(1).to_pandas()
if len(sample) > 0:
    obj = _json.loads(sample["FEATURES_OBJ"].iloc[0])
    print(f"\nFEATURES_OBJ keys: {list(obj.keys())}")
    print(f"cat_dev features: {len(obj.get('cat_dev', {}))} unique combinations")
    print(f"src_dev features: {len(obj.get('src_dev', {}))} unique combinations")

## 9. VECTOR Embeddings via Cortex

This Feature View uses `SNOWFLAKE.CORTEX.EMBED_TEXT_768` to generate semantic embeddings from product descriptions, stored as `VECTOR(FLOAT, 768)`. These can be used for similarity search, recommendation systems, or as features in downstream models.

In [ ]:
from feature_definitions.embedding_features import product_embeddings

fv = product_embeddings(session, "DEV")
fv_reg = fs.register_feature_view(feature_view=fv, version="V01", block=True)
print(f"PRODUCT_EMBEDDINGS V01: {fv_reg.status}")

# Preview
sample = fv_reg.feature_df.limit(1).to_pandas()
if len(sample) > 0:
    emb = sample["EMBEDDING"].iloc[0]
    print(f"Embedding type: {type(emb)}")
    print(f"Embedding dimensions: {len(emb) if hasattr(emb, '__len__') else 'scalar'}")

## 10. Feature View Summary

All 13 Feature Views are now registered across the development Feature Store.

In [ ]:
fvs = fs.list_feature_views().to_pandas()
print(f"Total Feature Views: {len(fvs)}\n")

for _, row in fvs.iterrows():
    ftype = "Tiled" if row.get("REFRESH_FREQ") and "hour" in str(row["REFRESH_FREQ"]) else \
            "DT" if row.get("REFRESH_FREQ") else "View"
    print(f"  {row['NAME']:40s} {row['VERSION']:5s} {ftype:6s} refresh={row['REFRESH_FREQ']}")

## 11. Enable ROW_TIMESTAMP on Dynamic Tables

Enable `ROW_TIMESTAMP` on all DT-backed Feature Views so that `METADATA$ROW_LAST_COMMIT_TIME` is available for pipeline latency measurement. This is used in Notebook 05 to track data propagation delay from source tables through each DT tier.

In [ ]:
from feature_definitions.latency import enable_row_timestamp_on_dts

results = enable_row_timestamp_on_dts(session, "DEV")
for r in results:
    print(r)

## Feature-to-Model Matrix

| Feature View | Entity | Type | Conversion | Churn | Notes |
|---|---|---|:---:|:---:|---|
| USER_PROFILE_FEATURES | USER | View | ✓ | ✓ | Shared |
| USER_PURCHASE_AGGREGATES | USER | Tiled | ✓ | ✓ | Shared, 7d/30d windows |
| USER_SESSION_ENGAGEMENT | USER | Tiled | ✓ | ✓ | Shared, event counts |
| SESSION_BEHAVIOR_FEATURES | SESSION | DT | ✓ | | Conversion-specific |
| USER_ENGAGEMENT_REALTIME | USER | Tiled | ✓ | | Hourly granularity |
| PRODUCT_CATALOG_FEATURES | PRODUCT | View | ✓ | | Product dimensions |
| USER_RECENCY_RAW | USER | DT | | ✓ | Raw timestamps (PIT-correct) |
| USER_RECENCY_FEATURES | USER | View | | ✓ | DAYS_SINCE derivation (serving) |
| USER_TREND_FEATURES | USER | Tiled | | ✓ | 90d windows |
| PRODUCT_SUPPLIER_METRICS | PRODUCT_SUPPLIER | View | | | Compound entity demo |
| USER_ORDER_SUMMARY_DBT | USER | View | | | External pipeline demo |
| USER_PERMUTATION_FEATURES_WIDE | USER | DT | ✓ | | OBJECT encapsulation |
| PRODUCT_EMBEDDINGS | PRODUCT | DT | ✓ | | VECTOR embeddings |

Proceed to **Notebook 02: ML Development** to use these features for model training.